# ALADIN workflow
This Notebook follow the step-by-step workflow for HW-SW co-design with the proposed framework.

### Imports

In [1]:
import os
import json
from importlib import import_module
from pathlib import Path

p = Path(__file__).resolve() if "__file__" in globals() else Path.cwd()

while not (p / "ALADIN").exists():   # or any marker file
    if p.parent == p:
        raise RuntimeError("Project root not found")
    p = p.parent
os.chdir(os.path.join(p, "ALADIN"))
print("CWD set to project root:", p)

MODEL_DIR = "./models"

CWD set to project root: /workspaces


## 1. Implementation-Aware model

In [2]:
import onnx
from qonnx.core.modelwrapper import ModelWrapper

target_model = "resnet_4_bit_mix_impl.onnx"
model_path = os.path.join(MODEL_DIR, target_model)
mw = ModelWrapper(onnx.load(model_path))

## 2. Platform-Aware modele
TODO: Graham's bound

## 3. Code Generation with Dory
TODO: multi-LUT implementation

In [3]:
FRONTEND = "QONNX"                  # "NEMO", "Quantlab", "QONNX"
HARDWARE = "PULP.PULP_gvsoc"        # "PULP.GAP8", "PULP.GAP8_L2", "PULP.PULP_gvsoc", "PULP.GAP9", "PULP.GAP9_NE16"
VERBOSE = "Check_all+Perf_final"    # "None", "Perf_final", "Check_all+Perf_final", "Last+Perf_final"
ISA = "mixed-sw"                    # "auto", "8bit", "mixed-hw", "mixed-sw"

# default config files
dory_config = {
	"BNRelu_bits": 32,
	"code reserved space": 150000,
    "input_bits": 8,
    "input_signed": False
}

In [ ]:
onnx_manager = import_module(f"dory.Frontend_frameworks.{FRONTEND}.Parser")
onnx_to_dory = onnx_manager.onnx_manager

graph = onnx_to_dory(
    model_path, 
    dory_config, 
    delta=2**16
).full_graph_parsing()


###################################
## DORY GENERAL PARSING OF QONNX ##
## FINAL RAPRESENTATION: DORY IR ##
###################################


Before: Gemm Gemm_0 inputs: ['Quant_71_out0', '436pdI', 'OXc309_bias']
After: Gemm Gemm_0 inputs: ['4UEs7F', '436pdI', 'OXc309_bias']
Before: GlobalAveragePool GlobalAveragePool_0 inputs: ['Quant_70_out0']
After: GlobalAveragePool GlobalAveragePool_0 inputs: ['xoKicF']
Before: Add Add_7 inputs: ['Quant_69_out0', 'Quant_67_out0']
After: Add Add_7 inputs: ['SgQqx5', 'Quant_67_out0']
Before: Conv Conv_19 inputs: ['Quant_68_out0', 'nMLUGs', '0p0Sxf_bias']
After: Conv Conv_19 inputs: ['HETW6u', 'nMLUGs', '0p0Sxf_bias']
Before: Conv Conv_18 inputs: ['Quant_67_out0', 'XwQNby', 'CXWInc_bias']
After: Conv Conv_18 inputs: ['Vw65hI', 'XwQNby', 'CXWInc_bias']
Before: Add Add_7 inputs: ['SgQqx5', 'Quant_67_out0']
After: Add Add_7 inputs: ['SgQqx5', 'Vw65hI']
Before: Add Add_6 inputs: ['Quant_66_out0', 'Quant_64_out0']
After: Add Add_6 inputs: ['j27GCB', 'Quant_64_out0']
Before: Conv Conv_17 inputs: ['Quant_65_out0', '2gRBkx', 'Y2Rk5i_bias']
After: Conv Conv_17 inputs: ['EAQdNg', '2gRBkx', 'Y2Rk5i_b

In [ ]:
from copy import deepcopy
L1_capacity, L2_capacity = 64000, 512000

onnx_manager = import_module(f'dory.Hardware_targets.{HARDWARE}.HW_Parser')
dory_to_dory_hw = onnx_manager.onnx_manager
graph_tmp = deepcopy(graph)
hw_graph = dory_to_dory_hw(
    graph_tmp,
    n_inputs=1,
    verify_checksum=False,
    L1_capacity=L1_capacity,
    L2_capacity=L2_capacity,
    config_file=dory_config
).full_graph_parsing()

onnx_manager_PULP: Key 'double_buffering' not found in HW_description.json - setting to 2
#####################################################
## DORY GENERAL PARSING FROM DORY IR TO DORY HW IR ##
## FINAL RAPRESENTATION: DORY HW IR                ##
#####################################################
Creating 00_DORY_HW_input_graph.json in logs/HW_related/json_files/
Creating 00_DORY_HW_input_graph.onnx in logs/HW_related/onnx_files/

Backend: Matching patterns from generated DORY ONNX to HW Nodes.
Creating 01_DORY_HW_graph_raw.json in logs/HW_related/json_files/
Creating 01_DORY_HW_graph_raw.onnx in logs/HW_related/onnx_files/

DORY generic Frontend. Updating branches pointers.
Creating 02_DORY_HW_graph_fixed_branches.json in logs/HW_related/json_files/
Creating 02_DORY_HW_graph_fixed_branches.onnx in logs/HW_related/onnx_files/

Updating dimensions of vectors inside the graph, if they do not match among nodes
Creating 03_DORY_HW_graph_fixed_dimensions.json in logs/HW_related/json

/workspaces/ALADIN/dory/Parsers/HW_node.py:184: RuntimeWarning: overflow encountered in scalar add
  self.check_sum_w += sum(self.__dict__[weight_name]["value"])
/workspaces/ALADIN/dory/Parsers/HW_node.py:203: RuntimeWarning: overflow encountered in scalar add
  self.check_sum_w += sum(self.__dict__[bias_name]["value"])


In [ ]:
onnx_manager = import_module(f'dory.Hardware_targets.{HARDWARE}.C_Parser')
dory_hw_to_c = onnx_manager.C_Parser
dory_hw_to_c(
    hw_graph, 
    dory_config,
    verbose_level=VERBOSE,
    perf_layer="Yes", 
    precision_library=ISA,
    app_directory="./application", 
    model_dir="./models",
    n_inputs=1,
    L1_capacity=L1_capacity,
    L2_capacity=L2_capacity
).full_graph_parsing()

C_Parser_PULP: Key 'double_buffering' not found in HW_description.json - setting to 2
#####################################################
## DORY GENERAL PARSING FROM DORY HW IR TO C FILES ##
## FINAL RAPRESENTATION: COMPILABLE C PROJECT      ##
#####################################################

Generating the .c file of the network.

Generating the Makefile.
Makefile template writer: key 'single_core_dma' not found in HW description, using multi-core transfers!
Makefile template writer: key 'single_core_dma' not found in HW description, using multi-core transfers!

Mapping the layers files to their templates and copying the kernels associated.

Copying Utils.

Generating .hex weight files.

Generating .hex input file.
Done!


In [ ]:
with open("./logs/HW_related/json_files/06_DORY_HW_tiled_graph.json", "r") as f:
    layers_tiling_info = json.load(f)["graph"]

## 4. GVSoC simulation
*NOTE*: this portion of the notebooks run only inside the container.

In [ ]:
import subprocess
NUM_CORES = 8


cmd = cmd = """
source .devcontainer/pulp_sdk.sh
export PATH=/dory_env/bin:$PATH
make -C ./application clean all run platform=gvsoc CORE=8
"""

try:
    proc = subprocess.run(
        ["bash", "-c", cmd],
        check=True,
        capture_output=True,
        text=True,
        timeout=720,
    )
    
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        f"Build or run failed (exit {e.returncode}):\n"
        f"STDOUT:\n{e.stdout}\n"
        f"STDERR:\n{e.stderr}"
    )
except subprocess.TimeoutExpired as e:
    raise TimeoutError(
        f"Test timed out after 720s.\n"
        f"Partial STDOUT:\n{e.stdout}\n"
        f"STDERR:\n{e.stderr}"
    )
    
proc.stdout

"make: Entering directory '/workspaces/ALADIN/application'\nRM  /workspaces/ALADIN/application/BUILD/PULP/GCC_RISCV/\nCC  src/ReluConvolution0.c\nCC  src/mem.c\nCC  src/main.c\nCC  src/dory_dma.c\nCC  src/pulp_nn_matmul_i8_u8_i8.c\nCC  src/pulp_nn_linear_u8_i8_i8.c\nCC  src/pulp_nn_matmul_u8_i8_i8.c\nCC  src/network.c\nCC  src/net_utils.c\nCC  src/pulp_nn_conv_i8_u8_i8.c\nCC  src/ReluFullyConnected1.c\nCC  /pulp-sdk/rtos/pulpos/common/kernel/fll-v1.c\nCC  /pulp-sdk/rtos/pulpos/common/kernel/freq-domains.c\nCC  /pulp-sdk/rtos/pulpos/pulp/kernel/chips/pulp/soc.c\nCC  /pulp-sdk/rtos/pmsis/pmsis_bsp/ram/ram.c\nCC  /pulp-sdk/rtos/pmsis/pmsis_bsp/ram/alloc_extern.c\nCC  /pulp-sdk/rtos/pmsis/pmsis_bsp/ram/hyperram/hyperram.c\nCC  /pulp-sdk/rtos/pmsis/pmsis_bsp/fs/read_fs/read_fs.c\nCC  /pulp-sdk/rtos/pmsis/pmsis_bsp/fs/host_fs/semihost.c\nCC  /pulp-sdk/rtos/pmsis/pmsis_bsp/fs/host_fs/host_fs.c\nCC  /pulp-sdk/rtos/pmsis/pmsis_bsp/fs/fs.c\nCC  /pulp-sdk/rtos/pmsis/pmsis_bsp/flash/hyperflash/hyp

Store results:

In [ ]:
import csv

csv_path = os.path.join(
    "./output",
    f"perf_{NUM_CORES}core_{L1_capacity//1000}L1_{L2_capacity//1000}L2_{target_model.split('.')[0]}.csv"
)
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

In [ ]:

def sum_componenets(tile: dict) -> int:
    return sum((
        tile["weight_memory"],
        tile["bias_memory"],
        tile["constants_memory"],
        tile["input_activation_memory"],
        tile["output_activation_memory"],
        tile.get("lut_memory", 0),
    ))
                        


with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "layer_name", 
        "MACs", 
        "num_cycles", 
        "MAC_per_cycle", 
        "num_cores", 
        "L1_mem", 
        "L2_mem", 
        "L1_tiling", 
        "L2_tiling"
    ])
    idx = 0
    for line in proc.stdout.splitlines():
        if not line.startswith("PERF_LOG"):
            continue
            
        info = line.strip().split(",")
        if len(info) != 6:
            continue
        
        _, layer_name, macs, cycles, perf, cores = info
        tiling_info = layers_tiling_info[idx]["Tiling_parameters"]
        idx += 1
        writer.writerow([
            layer_name,
            int(macs),
            int(cycles),
            float(perf),
            int(cores),
            L1_capacity,
            L2_capacity,
            sum_componenets(tiling_info["L1"]),
            sum_componenets(tiling_info["L2"])
        ])